# 10 Build Exposure and Vulnerability Features

Stage: `02_feature_engineering`
Discipline: social risk and baseline feature engineering.

Primary dependencies:
- `JupyterNotebooks/outputs/census_pr/municipio_risk_features.csv`
- `JupyterNotebooks/outputs/census_pr/town_risk_features.csv`

Outputs:
- `JupyterNotebooks/outputs/index_pipeline/10_features/municipio_exposure_vulnerability_features.csv`
- `JupyterNotebooks/outputs/index_pipeline/10_features/municipio_adjustment_factors.csv`
- `JupyterNotebooks/outputs/index_pipeline/10_features/adjustment_factor_reference.csv`


In [1]:
# Cell 1: Setup
import importlib.util
import subprocess
import sys
from pathlib import Path
import logging


def ensure_packages(packages):
    missing = [p for p in packages if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


ensure_packages(["pandas", "numpy", "pyyaml"])

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("index-pipeline-stage10")


def find_repo_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "JupyterNotebooks").exists():
            return c
    return p


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from scripts.factors.social_adjustments import (
    apply_age_adjustment,
    build_adjustment_factor_reference,
    build_municipio_adjustment_table,
    load_social_adjustment_spec,
    robust_score,
)

CENSUS_DIR = REPO_ROOT / "JupyterNotebooks" / "outputs" / "census_pr"
OUTPUT_DIR = REPO_ROOT / "JupyterNotebooks" / "outputs" / "index_pipeline" / "10_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOCIAL_ADJUSTMENT_SPEC = load_social_adjustment_spec(REPO_ROOT)

try:
    from IPython.display import display
except ImportError:
    display = print


In [2]:
# Cell 2: Load census-derived base files
muni_file = CENSUS_DIR / "municipio_risk_features.csv"
town_file = CENSUS_DIR / "town_risk_features.csv"

if not muni_file.exists():
    raise FileNotFoundError(f"Missing required file: {muni_file}")
if not town_file.exists():
    raise FileNotFoundError(f"Missing required file: {town_file}")

muni_df = pd.read_csv(muni_file)
town_df = pd.read_csv(town_file)

# municipio centroids available in town_risk_features output
town_coords = town_df[["municipio_key", "municipio", "latitude", "longitude"]].drop_duplicates("municipio_key")

base = muni_df.copy()
if "municipio_key" not in base.columns:
    base["municipio_key"] = base["municipio"].astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)

base = base.merge(town_coords, on=["municipio_key", "municipio"], how="left")

print(f"Municipios loaded: {len(base)}")
display(base.head(10))


Municipios loaded: 78


,NAME,population,median_income,poverty_universe,poverty_count,housing_units,occupied_units,vacant_units,no_vehicle_owner,no_vehicle_renter,...,elderly_population,elderly_rate,children_under_5,children_rate,age_vulnerable_rate,score_age_vulnerability,risk_index_raw,risk_index,latitude,longitude
0,"Adjuntas Municipio, Puerto Rico",17960,19549,17925,10915,7937,6069,1868,344,349,...,4098,0.228174,538,0.029955,0.258129,0.458749,0.484756,48.5,18.162560,-66.723136
1,"Aguada Municipio, Puerto Rico",37758,21501,37602,17495,17824,13100,4724,1019,619,...,8758,0.231951,1066,0.028232,0.260183,0.475445,0.435301,43.5,18.379773,-67.188754
2,"Aguadilla Municipio, Puerto Rico",54137,21540,53371,23929,28777,22295,6482,1372,1953,...,13296,0.245599,1656,0.030589,0.276188,0.605533,0.474978,47.5,18.429792,-67.154220
3,"Aguas Buenas Municipio, Puerto Rico",23614,23072,23525,10693,10959,8932,2027,551,660,...,5428,0.229864,698,0.029559,0.259422,0.469260,0.406934,40.7,18.256724,-66.103005
4,"Aibonito Municipio, Puerto Rico",24636,21744,24604,11460,10919,9777,1142,1089,456,...,6151,0.249675,723,0.029347,0.279023,0.628571,0.451829,45.2,18.139311,-66.266396
5,"Añasco Municipio, Puerto Rico",25094,24436,24978,10727,12640,9524,3116,463,404,...,5876,0.234160,673,0.026819,0.260979,0.481910,0.357456,35.7,18.282298,-67.140197
6,"Arecibo Municipio, Puerto Rico",86467,23635,85187,35141,43237,33403,9834,1855,2429,...,20827,0.240866,2664,0.030809,0.271676,0.568857,0.444926,44.5,18.472578,-66.715515
7,"Arroyo Municipio, Puerto Rico",15341,21186,15294,7974,8385,6739,1646,574,281,...,3451,0.224953,536,0.034939,0.259892,0.473076,0.438020,43.8,17.964269,-66.061702
8,"Barceloneta Municipio, Puerto Rico",22494,24508,22459,9421,10798,9053,1745,579,215,...,4828,0.214635,740,0.032898,0.247533,0.372620,0.315528,31.6,18.454664,-66.538688
9,"Barranquitas Municipio, Puerto Rico",29054,23407,28977,14057,11843,10014,1829,633,291,...,5629,0.193743,1093,0.037620,0.231362,0.241187,0.342723,34.3,18.186253,-66.306594


In [3]:
# Cell 3: Engineer exposure/vulnerability features
feat = base.copy()

# Exposure components
feat["exposure_population"] = pd.to_numeric(feat.get("population"), errors="coerce")
feat["exposure_population_score"] = robust_score(feat["exposure_population"], higher_is_worse=True)

# Vulnerability components
feat["poverty_rate"] = pd.to_numeric(feat.get("poverty_rate"), errors="coerce")
feat["no_vehicle_rate"] = pd.to_numeric(feat.get("no_vehicle_rate"), errors="coerce")
feat["vacancy_rate"] = pd.to_numeric(feat.get("vacancy_rate"), errors="coerce")
feat["median_income"] = pd.to_numeric(feat.get("median_income"), errors="coerce")

feat["poverty_score"] = robust_score(feat["poverty_rate"], higher_is_worse=True)
feat["transport_constraint_score"] = robust_score(feat["no_vehicle_rate"], higher_is_worse=True)
feat["housing_fragility_score"] = robust_score(feat["vacancy_rate"], higher_is_worse=True)
feat["income_capacity_score"] = robust_score(feat["median_income"], higher_is_worse=False)

feat["vulnerability_score_base"] = (
    0.35 * feat["poverty_score"]
    + 0.25 * feat["transport_constraint_score"]
    + 0.15 * feat["housing_fragility_score"]
    + 0.25 * (100 - feat["income_capacity_score"])
)
feat = apply_age_adjustment(
    feat,
    SOCIAL_ADJUSTMENT_SPEC,
    base_score_column="vulnerability_score_base",
    adjusted_score_column="vulnerability_score_adjusted",
)
feat["vulnerability_score"] = feat["vulnerability_score_adjusted"]
feat["exposure_score"] = feat["exposure_population_score"]

# Optional resilience baseline proxy
feat["resilience_capacity_score"] = (
    0.45 * feat["income_capacity_score"]
    + 0.30 * (100 - feat["transport_constraint_score"])
    + 0.25 * (100 - feat["housing_fragility_score"])
).clip(0, 100)

adjustment_table = build_municipio_adjustment_table(feat, SOCIAL_ADJUSTMENT_SPEC)
adjustment_reference = build_adjustment_factor_reference(SOCIAL_ADJUSTMENT_SPEC)

out_cols = [
    "municipio", "municipio_key", "latitude", "longitude",
    "population", "median_income", "poverty_rate", "no_vehicle_rate", "vacancy_rate",
    "child_under_5_population", "elderly_65_plus_population", "child_rate", "elderly_65_plus_rate",
    "exposure_score", "vulnerability_score_base", "vulnerability_score_adjusted", "vulnerability_score",
    "resilience_capacity_score", "poverty_score", "transport_constraint_score",
    "housing_fragility_score", "income_capacity_score", "score_child_vulnerability",
    "score_elderly_vulnerability", "score_age_vulnerability", "age_adjustment_points",
    "age_adjustment_enabled", "adjustment_config_version"
]

feat_out = feat[out_cols].copy()
out_file = OUTPUT_DIR / "municipio_exposure_vulnerability_features.csv"
adjustment_out = OUTPUT_DIR / SOCIAL_ADJUSTMENT_SPEC["output"]["municipio_adjustment_filename"]
reference_out = OUTPUT_DIR / SOCIAL_ADJUSTMENT_SPEC["output"]["factor_reference_filename"]
feat_out.to_csv(out_file, index=False)
adjustment_table.to_csv(adjustment_out, index=False)
adjustment_reference.to_csv(reference_out, index=False)

print("Outputs written:")
print(f"- {out_file}")
print(f"- {adjustment_out}")
print(f"- {reference_out}")
display(feat_out.head(10))
display(adjustment_table.head(10))


Outputs written:
- /Users/sivamanisubrahmanyaharivamsipullipudi/Downloads/DEAN_690/Spring2026DEAN/Spring2026DAEN/JupyterNotebooks/JupyterNotebooks/outputs/index_pipeline/10_features/municipio_exposure_vulnerability_features.csv
- /Users/sivamanisubrahmanyaharivamsipullipudi/Downloads/DEAN_690/Spring2026DEAN/Spring2026DAEN/JupyterNotebooks/JupyterNotebooks/outputs/index_pipeline/10_features/municipio_adjustment_factors.csv
- /Users/sivamanisubrahmanyaharivamsipullipudi/Downloads/DEAN_690/Spring2026DEAN/Spring2026DAEN/JupyterNotebooks/JupyterNotebooks/outputs/index_pipeline/10_features/adjustment_factor_reference.csv


,municipio,municipio_key,latitude,longitude,population,median_income,poverty_rate,no_vehicle_rate,vacancy_rate,child_under_5_population,...,poverty_score,transport_constraint_score,housing_fragility_score,income_capacity_score,score_child_vulnerability,score_elderly_vulnerability,score_age_vulnerability,age_adjustment_points,age_adjustment_enabled,adjustment_config_version
0,Adjuntas,adjuntas,18.162560,-66.723136,17960,19549,0.608926,0.114187,0.235353,NaN,...,100.000000,26.549313,44.200586,92.911071,NaN,48.368831,29.021299,3.482556,True,1.0
1,Aguada,aguada,18.379773,-67.188754,37758,21501,0.465268,0.125038,0.265036,NaN,...,66.994429,37.688095,56.644840,82.196717,NaN,53.609679,32.165807,3.859897,True,1.0
2,Aguadilla,aguadilla,18.429792,-67.154220,54137,21540,0.448352,0.149137,0.225249,NaN,...,60.927416,62.424903,39.964497,81.982650,NaN,65.800801,39.480481,4.737658,True,1.0
3,Aguas Buenas,aguas buenas,18.256724,-66.103005,23614,23072,0.454538,0.135580,0.184962,NaN,...,63.145962,48.509129,23.074269,73.573638,NaN,35.980956,21.588573,2.590629,True,1.0
4,Aibonito,aibonito,18.139311,-66.266396,24636,21744,0.465778,0.158024,0.104588,NaN,...,67.177389,71.547696,0.000000,80.862912,NaN,74.869603,44.921762,5.390611,True,1.0
5,Añasco,anasco,18.282298,-67.140197,25094,24436,0.429458,0.091033,0.246519,NaN,...,54.150799,2.782271,48.881705,66.086763,NaN,46.644880,27.986928,3.358431,True,1.0
6,Arecibo,arecibo,18.472578,-66.715515,86467,23635,0.412516,0.128252,0.227444,NaN,...,48.074379,40.987018,40.884638,70.483381,NaN,51.618268,30.970961,3.716515,True,1.0
7,Arroyo,arroyo,17.964269,-66.061702,15341,21186,0.521381,0.126873,0.196303,NaN,...,87.120055,39.571969,27.828845,83.925724,NaN,40.274324,24.164594,2.899751,True,1.0
8,Barceloneta,barceloneta,18.454664,-66.538688,22494,24508,0.419475,0.087706,0.161604,NaN,...,50.570483,0.000000,13.281478,65.691562,NaN,23.475633,14.085380,1.690246,True,1.0
9,Barranquitas,barranquitas,18.186253,-66.306594,29054,23407,0.485109,0.092271,0.154437,NaN,...,74.110661,4.052699,10.276837,71.734853,NaN,6.747815,4.048689,0.485843,True,1.0


,municipio,municipio_key,municipio_slug,child_under_5_population,elderly_65_plus_population,child_rate,elderly_65_plus_rate,age_dependency_rate,score_child_vulnerability,score_elderly_vulnerability,score_age_vulnerability,age_adjustment_points,age_adjustment_enabled,vulnerability_score_base,vulnerability_score_adjusted,adjustment_config_version
0,Adjuntas,adjuntas,adjuntas,NaN,3659.0,NaN,0.203731,0.203731,NaN,48.368831,29.021299,3.482556,True,50.039648,53.522204,1.0
1,Aguada,aguada,aguada,NaN,7827.0,NaN,0.207294,0.207294,NaN,53.609679,32.165807,3.859897,True,45.817621,49.677518,1.0
2,Aguadilla,aguadilla,aguadilla,NaN,11671.0,NaN,0.215583,0.215583,NaN,65.800801,39.480481,4.737658,True,47.429833,52.167491,1.0
3,Aguas Buenas,aguas buenas,aguas buenas,NaN,4612.0,NaN,0.195308,0.195308,NaN,35.980956,21.588573,2.590629,True,44.296100,46.886729,1.0
4,Aibonito,aibonito,aibonito,NaN,5463.0,NaN,0.221749,0.221749,NaN,74.869603,44.921762,5.390611,True,46.183282,51.573894,1.0
5,Añasco,anasco,anasco,NaN,5083.0,NaN,0.202558,0.202558,NaN,46.644880,27.986928,3.358431,True,35.458912,38.817344,1.0
6,Arecibo,arecibo,arecibo,NaN,17807.0,NaN,0.205940,0.205940,NaN,51.618268,30.970961,3.716515,True,40.584638,44.301153,1.0
7,Arroyo,arroyo,arroyo,NaN,3041.0,NaN,0.198227,0.198227,NaN,40.274324,24.164594,2.899751,True,48.577907,51.477659,1.0
8,Barceloneta,barceloneta,barceloneta,NaN,4202.0,NaN,0.186805,0.186805,NaN,23.475633,14.085380,1.690246,True,28.269000,29.959246,1.0
9,Barranquitas,barranquitas,barranquitas,NaN,5097.0,NaN,0.175432,0.175432,NaN,6.747815,4.048689,0.485843,True,35.559719,36.045561,1.0
